In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("silver_table", "environment")

In [0]:
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
silver_table_name = dbutils.widgets.get("silver_table")

silver_table = f"{catalog}.{silver_schema}.{silver_table_name}"

In [0]:
from pyspark.sql import functions as F

In [0]:
print("Rows:", spark.table(silver_table).count())

duplicate_count = (
    spark.table(silver_table)
    .groupBy("AREA", "YEAR") # for yield_and_climate ["AREA", "ITEM", "YEAR"]
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate keys:", duplicate_count)

###Table layout and optimization

In [0]:
optimize_result = spark.sql(f"""OPTIMIZE {silver_table}""")

In [0]:
spark.sql(f"""VACUUM {silver_table}""")

####Liquid Clustering vs ZORDER vs Partitioning

**Liquid Clustering **dynamically organizes data by selected columns and is the recommended approach for new Delta tables.
**ZORDER** improves data skipping for selected columns but usually requires repeated **OPTIMIZE ... ZORDER BY**.
**Partitioning** physically separates data by partition columns and works best for large tables with low-cardinality columns.

###Data quality rules:

In [0]:
df = spark.table(silver_table)

quarantine_table = f"{catalog}.{silver_schema}.environment_quarantine"

rules = (
    F.col("AREA").isNotNull()
    & F.col("YEAR").isNotNull()   & (F.col("YEAR") >= 1900)
    & (F.col("freshwater_per_capita").isNull() | (F.col("freshwater_per_capita") >= 0))
    & (F.col("freshwater_withdrawals").isNull() | (F.col("freshwater_withdrawals") >= 0))
    & (F.col("agricultural_land").isNull() | (F.col("agricultural_land") >= 0))
    & (F.col("forest_area").isNull() | (F.col("forest_area") >= 0)))

df_valid = df.filter(rules)
df_invalid = df.filter(~rules)

In [0]:
(
    df_valid.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

if df_invalid.limit(1).count() > 0:
    (
        df_invalid.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(quarantine_table)
    )

In [0]:
print("Valid rows:", df_valid.count())
print("Invalid rows:", df_invalid.count())